# [5.3] Mamba from Scratch - Exercises

This section builds the smallest useful Mamba implementation contract. The point is not to memorize another block diagram; it is to make every state-space execution path agree with a simple recurrent reference before trusting a fast kernel or a cached generation loop.

<img src="../../instructions/assets/mamba_scan_contract.svg" width="820">

The exercises move in this order:

1. discretize stable continuous-time SSM parameters,
2. implement the recurrent selective scan,
3. implement an associative scan and a chunked scan that match the recurrent path,
4. build the tiny block cache with convolution and SSM state,
5. wrap the block as a tiny causal LM and verify recurrent-state cache parity,
6. inspect the committed CUDA preflight for the pinned Mamba-130M-HF checkpoint.


In [ ]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import torch as t
import torch.nn as nn
import torch.nn.functional as F

chapter = "chapter5_modern_architectures"
section = "part3_mamba_from_scratch"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part3_mamba_from_scratch.tests as tests
import part3_mamba_from_scratch.utils as utils
from part3_mamba_from_scratch.solutions import (
    MambaCausalLMOutput,
    MambaConfig,
    MambaInferenceState,
    MambaScanReport,
    mamba_cache_parity_report,
    scan_equivalence_report,
)

MAIN = __name__ == "__main__"


## 1. Synthetic Scan Inputs

Mamba's selective scan operates on the post-convolution stream `u`, input-dependent timestep `delta`, stable dynamics `A`, and input-dependent readout terms `B` and `C`. We use tiny generated tensors so the recurrence can be inspected directly.

<details><summary>Expected output</summary>

Running the cell should create tensors with shapes:

```python
u.shape      == (2, 7, 4)
delta.shape  == (2, 7, 4)
A_log.shape  == (4, 3)
B.shape      == (2, 7, 3)
C.shape      == (2, 7, 3)
```

</details>

<details><summary>Help - what the axes mean</summary>

`d_inner` is the expanded channel dimension inside a Mamba block. `d_state` is the size of the SSM state attached to each inner channel. The recurrent state therefore has shape `(batch, d_inner, d_state)`.

</details>


In [ ]:
def make_scan_inputs(seed: int = 0):
    t.manual_seed(seed)
    batch, seq, d_inner, d_state = 2, 7, 4, 3
    u = t.randn(batch, seq, d_inner)
    delta = t.rand(batch, seq, d_inner) + 0.1
    A_log = t.randn(d_inner, d_state) - 2.0
    B = t.randn(batch, seq, d_state)
    C = t.randn(batch, seq, d_state)
    D = t.randn(d_inner)
    z = t.randn(batch, seq, d_inner)
    return u, delta, A_log, B, C, D, z

u, delta, A_log, B, C, D, z = make_scan_inputs()
[u.shape, delta.shape, A_log.shape, B.shape, C.shape, D.shape, z.shape]


## 2. Discretize the Selective Scan

The stable continuous-time dynamics are parameterized as `A = -exp(A_log)`. Positive `delta` then gives a discrete decay coefficient in `(0, 1]`:

```text
a_t = exp(delta_t * A)
b_t = delta_t * u_t * B_t
```

<details><summary>Expected output</summary>

The test should print:

```text
All tests in `test_discretize_selective_scan_shapes_and_stability` passed!
```

It also checks that every discrete decay coefficient is positive and no larger than one.

</details>

<details><summary>Help - broadcasting B</summary>

The tests pass `B` with shape `(batch, seq, d_state)`. The scan needs a separate state vector for each `d_inner` channel, so expand `B` to `(batch, seq, d_inner, d_state)` before multiplying by `u.unsqueeze(-1)`.

</details>

<details><summary>Solution</summary>

Use `_expand_bc` to accept either rank-3 shared `B/C` tensors or rank-4 per-channel tensors. Then compute `A = -exp(A_log)` and return `a, b` with shape `(batch, seq, d_inner, d_state)`.

</details>


In [ ]:
def _expand_bc(param: t.Tensor, d_inner: int) -> t.Tensor:
    """Expand B/C parameters to shape (batch, seq, d_inner, d_state)."""
    raise NotImplementedError()


def discretize_selective_scan(
    u: t.Tensor,
    delta: t.Tensor,
    A_log: t.Tensor,
    B: t.Tensor,
) -> tuple[t.Tensor, t.Tensor]:
    """Return coefficients for state_t = a_t * state_{t-1} + b_t."""
    raise NotImplementedError()


tests.test_discretize_selective_scan_shapes_and_stability(discretize_selective_scan)


## 3. Recurrent Scan and Readout

The recurrent implementation is the reference path. Keep it boring and explicit: loop over sequence positions, update the SSM state, store each state, then compute the readout.

<details><summary>Expected output</summary>

The tests should print:

```text
All tests in `test_selective_scan_single_step_manual` passed!
All tests in `test_recurrent_scan_matches_reference` passed!
```

The single-step fixture is deliberately hand-checkable: with zero initial state, the new state is just `b_t`.

</details>

<details><summary>Help - where D and z enter</summary>

`D` is a skip connection from `u` to the output, not part of the state update. `z` is a gate applied to the output after the state readout and `D * u` term have been added.

</details>

<details><summary>Solution</summary>

Compute all `a, b` coefficients first. Initialize state to zeros unless `initial_state` is supplied, update `state = a[:, pos] * state + b[:, pos]`, stack the stored states, then apply the `C`, `D`, and `silu(z)` readout.

</details>


In [ ]:
def _expand_c(C: t.Tensor, d_inner: int, dtype: t.dtype, device: t.device) -> t.Tensor:
    return _expand_bc(C.to(device=device, dtype=dtype), d_inner=d_inner)


def _readout(
    states: t.Tensor,
    u: t.Tensor,
    C: t.Tensor,
    D: t.Tensor | None,
    z: t.Tensor | None,
) -> t.Tensor:
    raise NotImplementedError()


def selective_scan_recurrent(
    u: t.Tensor,
    delta: t.Tensor,
    A_log: t.Tensor,
    B: t.Tensor,
    C: t.Tensor,
    D: t.Tensor | None = None,
    z: t.Tensor | None = None,
    initial_state: t.Tensor | None = None,
    return_last_state: bool = False,
) -> t.Tensor | tuple[t.Tensor, t.Tensor]:
    """Reference recurrent selective scan."""
    raise NotImplementedError()


tests.test_selective_scan_single_step_manual(selective_scan_recurrent)
tests.test_recurrent_scan_matches_reference(selective_scan_recurrent)


## 4. Associative and Chunked Scans

The associative scan composes affine recurrence transforms:

```text
(a2, b2) o (a1, b1) = (a2 * a1, a2 * b1 + b2)
```

The chunked scan is simpler: run recurrent scan on slices, but carry the last SSM state into the next slice.

<details><summary>Expected output</summary>

The tests should print:

```text
All tests in `test_selective_scan_equivalence` passed!
All tests in `test_chunked_scan_equivalence` passed!
```

The signature local numbers should be on the order of `1e-6` or smaller.

</details>

<details><summary>Help - the chunk-boundary bug</summary>

The most common wrong implementation is to restart the recurrent state at zero at every chunk. That can pass shape tests but fails long-context equivalence because token `chunk_start` cannot see the previous chunk's state.

</details>

<details><summary>Solution</summary>

For the associative path, maintain prefix-composed `a_prefix, b_prefix` tensors. For the chunked path, call the recurrent implementation with `return_last_state=True` and pass that state into the next chunk.

</details>


In [ ]:
def selective_scan_parallel(
    u: t.Tensor,
    delta: t.Tensor,
    A_log: t.Tensor,
    B: t.Tensor,
    C: t.Tensor,
    D: t.Tensor | None = None,
    z: t.Tensor | None = None,
    initial_state: t.Tensor | None = None,
    return_last_state: bool = False,
) -> t.Tensor | tuple[t.Tensor, t.Tensor]:
    """Associative-scan version of the selective scan."""
    raise NotImplementedError()


def selective_scan_chunked(
    u: t.Tensor,
    delta: t.Tensor,
    A_log: t.Tensor,
    B: t.Tensor,
    C: t.Tensor,
    D: t.Tensor | None = None,
    z: t.Tensor | None = None,
    chunk_size: int = 64,
    initial_state: t.Tensor | None = None,
    return_last_state: bool = False,
) -> t.Tensor | tuple[t.Tensor, t.Tensor]:
    """Run the recurrent scan in chunks while carrying SSM state."""
    raise NotImplementedError()


def selective_scan_equivalence_smoke_test() -> dict:
    u, delta, A_log, B, C, D, z = make_scan_inputs()
    recurrent = selective_scan_recurrent(u, delta, A_log, B, C, D=D, z=z)
    parallel = selective_scan_parallel(u, delta, A_log, B, C, D=D, z=z)
    return scan_equivalence_report(recurrent, parallel, atol=1e-5).__dict__


def chunked_scan_equivalence_smoke_test() -> dict:
    u, delta, A_log, B, C, D, z = make_scan_inputs()
    full = selective_scan_recurrent(u, delta, A_log, B, C, D=D, z=z)
    chunked = selective_scan_chunked(u, delta, A_log, B, C, D=D, z=z, chunk_size=3)
    return scan_equivalence_report(full, chunked, atol=1e-6).__dict__


tests.test_selective_scan_equivalence(selective_scan_equivalence_smoke_test)
tests.test_chunked_scan_equivalence(chunked_scan_equivalence_smoke_test)


## 5. Tiny Mamba Block

A Mamba block wraps the selective scan with an input projection, causal depthwise convolution, input-dependent scan parameters, a gate, and an output projection. The cache has two pieces:

- `conv_state`: the previous `d_conv - 1` projected tokens,
- `ssm_state`: the recurrent selective-scan state.

<details><summary>Expected output</summary>

The tests should print:

```text
All tests in `test_tiny_mamba_block_matches_reference` passed!
All tests in `test_block_step_equivalence` passed!
```

The block-level full-vs-step maximum absolute difference should be at most `1e-5`.

</details>

<details><summary>Help - convolution state is not SSM state</summary>

The causal convolution sees a short window of projected inputs before the SSM scan. The SSM state stores the recurrent state after discretization. Mixing these two states is the fastest route to a cache-parity bug.

</details>

<details><summary>Solution</summary>

Implement `_causal_conv` by left-padding before depthwise `Conv1d`. In `step`, concatenate the previous convolution window with the current projected token, compute one convolution output, update `conv_state`, then call the recurrent scan for one token using the previous `ssm_state`.

</details>


In [ ]:
def make_tiny_mamba_config() -> MambaConfig:
    return MambaConfig(
        vocab_size=29,
        d_model=12,
        d_inner=16,
        d_state=4,
        d_conv=3,
        dt_rank=4,
        num_layers=2,
    )


class MambaRMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.weight = nn.Parameter(t.ones(d_model))
        self.eps = eps

    def forward(self, x: t.Tensor) -> t.Tensor:
        raise NotImplementedError()


class TinyMambaBlock(nn.Module):
    """Minimal Mamba block with causal convolution and selective scan."""

    def __init__(self, config: MambaConfig):
        super().__init__()
        raise NotImplementedError()

    def initial_state(self, batch: int, device: t.device, dtype: t.dtype) -> MambaInferenceState:
        raise NotImplementedError()

    def _causal_conv(self, x: t.Tensor) -> t.Tensor:
        raise NotImplementedError()

    def _final_conv_state(self, x: t.Tensor) -> t.Tensor:
        raise NotImplementedError()

    def _scan_parameters(self, x: t.Tensor) -> tuple[t.Tensor, t.Tensor, t.Tensor]:
        raise NotImplementedError()

    def forward(
        self,
        hidden_states: t.Tensor,
        *,
        inference_state: MambaInferenceState | None = None,
        use_cache: bool = False,
    ) -> tuple[t.Tensor, MambaInferenceState | None]:
        raise NotImplementedError()

    def step(
        self,
        hidden_states: t.Tensor,
        inference_state: MambaInferenceState,
    ) -> tuple[t.Tensor, MambaInferenceState]:
        raise NotImplementedError()


def block_step_equivalence_smoke_test() -> dict:
    t.manual_seed(0)
    block = TinyMambaBlock(make_tiny_mamba_config())
    hidden = t.randn(2, 5, block.config.d_model)
    full, _ = block(hidden, use_cache=False)
    state = block.initial_state(batch=2, device=hidden.device, dtype=hidden.dtype)
    step_outputs = []
    for pos in range(hidden.shape[1]):
        out, state = block.step(hidden[:, pos : pos + 1], state)
        step_outputs.append(out)
    stepped = t.cat(step_outputs, dim=1)
    diff = (full - stepped).abs()
    return {"max_abs_diff": diff.max().item(), "passed": bool(diff.max().item() <= 1e-5)}


tests.test_tiny_mamba_block_matches_reference(TinyMambaBlock)
tests.test_block_step_equivalence(block_step_equivalence_smoke_test)


## 6. Tiny Mamba LM and Cache Parity

The LM wrapper is the Mamba analogue of transformer KV-cache parity. Full-sequence logits should match logits produced by feeding one token at a time while carrying per-layer recurrent states.

<details><summary>Expected output</summary>

The tests should print:

```text
All tests in `test_tiny_lm_matches_reference` passed!
All tests in `test_tiny_lm_cache_parity` passed!
All tests in `test_notebook_contract` passed!
```

The generated tensor shape should be `(1, 7)` after appending four tokens to a length-three prompt.

</details>

<details><summary>Help - tied embeddings</summary>

When `tie_word_embeddings=True`, the LM head should share storage with the token embedding matrix. The test checks pointer equality, not just equal values.

</details>

<details><summary>Solution</summary>

Store one `MambaInferenceState` per layer. During cached generation, call the model on the current token only, save the returned states, append `argmax` of the final logit, and feed that token into the next step.

</details>


In [ ]:
class TinyMambaModel(nn.Module):
    def __init__(self, config: MambaConfig):
        super().__init__()
        raise NotImplementedError()

    def initial_states(
        self,
        batch: int,
        device: t.device,
        dtype: t.dtype,
    ) -> tuple[MambaInferenceState, ...]:
        raise NotImplementedError()

    def forward(
        self,
        input_ids: t.Tensor,
        *,
        states: tuple[MambaInferenceState, ...] | None = None,
        use_cache: bool = False,
    ) -> tuple[t.Tensor, tuple[MambaInferenceState, ...] | None]:
        raise NotImplementedError()


class TinyMambaForCausalLM(nn.Module):
    def __init__(self, config: MambaConfig):
        super().__init__()
        raise NotImplementedError()

    def forward(
        self,
        input_ids: t.Tensor,
        *,
        states: tuple[MambaInferenceState, ...] | None = None,
        use_cache: bool = False,
    ) -> MambaCausalLMOutput:
        raise NotImplementedError()

    @t.no_grad()
    def greedy_generate(self, input_ids: t.Tensor, max_new_tokens: int) -> t.Tensor:
        raise NotImplementedError()


def tiny_lm_cache_parity_smoke_test() -> dict:
    t.manual_seed(1)
    model = TinyMambaForCausalLM(make_tiny_mamba_config())
    input_ids = t.tensor([[1, 2, 3, 4, 5]])
    return mamba_cache_parity_report(model, input_ids, atol=1e-5)


def generation_shape_smoke_test() -> tuple[int, ...]:
    t.manual_seed(2)
    model = TinyMambaForCausalLM(make_tiny_mamba_config())
    input_ids = t.tensor([[1, 2, 3]])
    return tuple(model.greedy_generate(input_ids, max_new_tokens=4).shape)


def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    return {
        "selective_scan_equivalence": selective_scan_equivalence_smoke_test(),
        "chunked_scan_equivalence": chunked_scan_equivalence_smoke_test(),
        "block_step_equivalence": block_step_equivalence_smoke_test(),
        "lm_cache_parity": tiny_lm_cache_parity_smoke_test(),
        "generation_shape": generation_shape_smoke_test(),
    }


tests.test_tiny_lm_matches_reference(TinyMambaForCausalLM)
tests.test_tiny_lm_cache_parity(tiny_lm_cache_parity_smoke_test)
tests.test_notebook_contract(run_smoke_test)


## Signature Result

The committed report checks the local implementation contract and then loads the pinned official Mamba checkpoint on CUDA. The goal is narrow: prove the scan/cache invariants and verify that the environment can run the real model path with the required fast kernels.

| Check | Current result | Acceptance rule |
|---|---:|---|
| Recurrent vs associative scan max diff | `4.77e-7` | `<= 1e-6` |
| Tiny LM cache max diff | `1.19e-6` | `<= 1e-5` |
| Official Mamba preflight | `true` | must pass |
| Fast kernels available | `true` | must pass |
| Batched/single top-1 agreement | `1.0` | exactly `1.0` |
| Generated new tokens | `4` | exactly `4` |
| Generation speed | `10.62 tokens/s` | `>= 1 token/s` |
| Peak VRAM | `0.279 GB` | below `24 GB` |

<details><summary>Interpreting the signature result</summary>

The result says the didactic implementation has the right execution invariants and that the pinned official checkpoint runs real logits and deterministic generation locally. It does not say the tiny implementation has loaded Mamba-130M weights or that every production Mamba kernel has been independently reimplemented.

</details>

<details><summary>Help - why the fast-kernel check is part of the evidence</summary>

Mamba is attractive partly because its recurrence can be executed efficiently. A course result that silently falls back to a slow or missing selective-scan kernel would be a different claim, so the report records whether Transformers can see the compiled Mamba fast path.

</details>


In [ ]:
def _load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return {
        "device": gpu["device"],
        "torch": gpu["torch_version"],
        "cuda": gpu["cuda_version"],
        "scan_max_abs_diff": gpu["scan_max_abs_diff"],
        "cache_max_abs_diff": gpu["cache_max_abs_diff"],
        "official_preflight": gpu["official_mamba_logits_generation_preflight_passed"],
        "fast_kernel_available": gpu["official_mamba_fast_kernel_available"],
        "batched_single_top1_agreement": gpu["official_mamba_batched_single_top1_agreement"],
        "generated_new_tokens": gpu["official_mamba_generation_new_tokens"],
        "generation_tokens_per_second": gpu["official_mamba_generation_tokens_per_second"],
        "peak_vram_gb": gpu["peak_vram_gb"],
    }


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


run_gpu_test()


## Limitations

This section proves the selective-scan and cache-parity contract for a didactic Mamba implementation and runs a pinned Mamba-130M-HF checkpoint preflight. It does not claim weight-level parity between the tiny model and the released checkpoint, benchmark long-context throughput, reproduce Mamba training, or audit Mamba internals for interpretability claims. The next section uses the recurrent state for model-organism state tracking.
